In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
import numpy as np
import pandas as pd
df_raw=pd.read_csv('/content/drive/MyDrive/Dataset/medical_cost_with_nan.csv')
df_raw.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.92400
1,18.0,male,NaN,1.0,no,southeast,1725.55230
2,28.0,male,33.000,3.0,no,southeast,NaN
3,33.0,male,22.705,0.0,no,northwest,21984.47061
4,32.0,male,28.880,0.0,NaN,northwest,3866.85520


In [2]:
df_raw.shape

(1338, 7)

In [3]:
df_raw.isnull().sum()

,0
age,66
sex,66
bmi,66
children,66
smoker,66
region,66
charges,66


In [4]:
df_raw = df_raw.dropna(subset=['charges'])
print(f"Shape of df_raw after dropping NaNs in 'charges': {df_raw.shape}")
print(df_raw.isnull().sum())

Shape of df_raw after dropping NaNs in 'charges': (1272, 7)
age         62
sex         63
bmi         65
children    62
smoker      63
region      62
charges      0
dtype: int64


In [5]:
feature=df_raw.drop('charges',axis=1)
label=df_raw['charges']
label.head()

,charges
0,16884.92400
1,1725.55230
3,21984.47061
4,3866.85520
5,3756.62160


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(feature,label,random_state=42,shuffle=True)
print(X_train.shape,y_train.shape)
print(X_test.shape,y_test.shape)

(954, 6) (954,)
(318, 6) (318,)


In [7]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 954 entries, 147 to 1179
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       912 non-null    float64
 1   sex       911 non-null    object 
 2   bmi       904 non-null    float64
 3   children  910 non-null    float64
 4   smoker    904 non-null    object 
 5   region    912 non-null    object 
dtypes: float64(3), object(3)
memory usage: 52.2+ KB


In [8]:
num_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='mean'))
])

In [9]:
cat_pipeline=Pipeline([
    ('imputer',SimpleImputer(strategy='constant',fill_value='Other'))
])

In [10]:
from sklearn.compose import ColumnTransformer,make_column_transformer,make_column_selector
preprocessor=ColumnTransformer(
    transformers=[
        ('num_imputer', num_pipeline, make_column_selector(dtype_include=np.float64)),
        ('cat_imputer', cat_pipeline, make_column_selector(dtype_include=object))
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
).set_output(transform='pandas')
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('num_imputer',
                                 Pipeline(steps=[('imputer', SimpleImputer())]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7c610077fd40>),
                                ('cat_imputer',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Other',
                                                                strategy='constant'))]),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7c60f13443b0>)],
                  verbose_feature_names_out=False)

In [11]:
X_train=preprocessor.fit_transform(X_train)
print(X_train.isnull().sum())
X_train.head()

age         0
bmi         0
children    0
sex         0
smoker      0
region      0
dtype: int64


,age,bmi,children,sex,smoker,region
147,51.0,37.73,1.000000,female,no,southeast
1141,41.0,32.60,3.000000,female,no,southwest
486,54.0,21.47,3.000000,female,no,northwest
1077,21.0,26.03,0.000000,male,no,northeast
145,29.0,38.83,1.064835,female,no,southeast


In [12]:
X_test=preprocessor.transform(X_test)
print(X_test.isnull().sum())
X_test.head()

age         0
bmi         0
children    0
sex         0
smoker      0
region      0
dtype: int64


,age,bmi,children,sex,smoker,region
216,53.0,26.600000,0.0,female,no,northwest
1008,25.0,24.985000,2.0,male,Other,northeast
739,29.0,35.500000,2.0,male,yes,southwest
1250,24.0,30.690487,0.0,Other,yes,northeast
602,56.0,25.300000,0.0,female,Other,southwest


In [13]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
# Corrected categorical attributes
cat_attr=['sex','smoker','region'] # Changed 'religion' to 'region'

# Use OrdinalEncoder for multiple categorical features in a pipeline
encoder_pipeline=Pipeline([
    ('cat_encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)), # Added handle_unknown for robustness
])
encoder_pipeline

Pipeline(steps=[('cat_encoder',
                 OrdinalEncoder(handle_unknown='use_encoded_value',
                                unknown_value=-1))])

In [14]:
preprocessor_cat=ColumnTransformer(transformers=[('categorical_encoding_scaling', encoder_pipeline, cat_attr)],remainder='passthrough',verbose_feature_names_out=False).set_output(transform='pandas')
preprocessor_cat

ColumnTransformer(remainder='passthrough',
                  transformers=[('categorical_encoding_scaling',
                                 Pipeline(steps=[('cat_encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['sex', 'smoker', 'region'])],
                  verbose_feature_names_out=False)

In [15]:
X_train=preprocessor_cat.fit_transform(X_train)
X_train.head()

,sex,smoker,region,age,bmi,children
147,1.0,1.0,3.0,51.0,37.73,1.000000
1141,1.0,1.0,4.0,41.0,32.60,3.000000
486,1.0,1.0,2.0,54.0,21.47,3.000000
1077,2.0,1.0,1.0,21.0,26.03,0.000000
145,1.0,1.0,3.0,29.0,38.83,1.064835


In [16]:
X_test=preprocessor_cat.transform(X_test)
X_test.head()

,sex,smoker,region,age,bmi,children
216,1.0,1.0,2.0,53.0,26.600000,0.0
1008,2.0,0.0,1.0,25.0,24.985000,2.0
739,2.0,2.0,4.0,29.0,35.500000,2.0
1250,0.0,2.0,1.0,24.0,30.690487,0.0
602,1.0,0.0,4.0,56.0,25.300000,0.0


In [17]:
from sklearn.linear_model import LinearRegression
final_pipeline=Pipeline([
    ('scaler',StandardScaler()),
    ('regression',LinearRegression())
])

In [21]:
final_pipeline.fit(X_train,y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('regression', LinearRegression())])

In [22]:
final_pipeline.predict(X_test[:5])

array([12622.08728792, -9808.36265934, 26750.26877369, 22724.28307697,
       -3500.00248099])

In [23]:
y_test[:5]

,charges
216,10355.64100
1008,23241.47453
739,44585.45587
1250,18648.42170
602,11070.53500


In [25]:
final_pipeline.score(X_test,y_test)

0.5997711198389513

In [26]:
final_pipeline.score(X_train,y_train)

0.5061276837374913